# Operaciones con Azure OpenAI

Este notebook contiene métodos para trabajar con Azure OpenAI (Azure AI):
- **Generar resumen** de un texto
- **Extraer puntos más relevantes** de un texto
- **Extraer métricas** como fecha de publicación y categorías de Alzheimer


## Instalación de dependencias

Primero, instalamos la biblioteca necesaria para trabajar con Azure OpenAI.


In [1]:
# Instalar openai si no está instalado
!pip install openai


## Importación de librerías


In [2]:
from openai import AzureOpenAI
from typing import List, Dict, Optional
import json
import re
from datetime import datetime


## Configuración de Azure OpenAI

Configuración necesaria para conectarse a Azure OpenAI.


In [3]:
# Configuración de Azure OpenAI
# Puedes obtener estos valores desde Azure Portal > Azure OpenAI > Keys and Endpoint
AZURE_OPENAI_ENDPOINT = "#"
AZURE_OPENAI_API_KEY = "#"
AZURE_OPENAI_API_VERSION = "#"  # O la versión más reciente disponible
AZURE_OPENAI_DEPLOYMENT_NAME = "#"  # Nombre del deployment en Azure

# Listado de tipos de Alzheimer para la extracción de categorías
TIPOS_ALZHEIMER = [
    "Alzheimer de inicio temprano",
    "Alzheimer de inicio tardío",
    "Alzheimer familiar",
    "Alzheimer esporádico",
    "Alzheimer con demencia vascular",
    "Alzheimer con cuerpos de Lewy",
    "Deterioro cognitivo leve (DCL)",
    "Demencia frontotemporal",
    "Demencia mixta"
]


## Método 1: Generar resumen

Este método genera un resumen conciso de un texto de entrada usando Azure OpenAI.


In [4]:
def generar_resumen(
    texto_entrada: str,
    endpoint: str = AZURE_OPENAI_ENDPOINT,
    api_key: str = AZURE_OPENAI_API_KEY,
    api_version: str = AZURE_OPENAI_API_VERSION,
    deployment_name: str = AZURE_OPENAI_DEPLOYMENT_NAME,
    max_tokens: int = 500
) -> str:
    """
    Genera un resumen conciso de un texto de entrada usando Azure OpenAI.
    
    Parámetros:
    -----------
    texto_entrada : str
        Texto del cual se desea generar un resumen
    endpoint : str
        Endpoint de Azure OpenAI
    api_key : str
        Clave API de Azure OpenAI
    api_version : str
        Versión de la API de Azure OpenAI
    deployment_name : str
        Nombre del deployment en Azure OpenAI
    max_tokens : int
        Número máximo de tokens para el resumen (por defecto 500)
    
    Retorna:
    --------
    str
        Resumen generado del texto de entrada
    
    Ejemplo:
    --------
    >>> resumen = generar_resumen(
    ...     texto_entrada="Texto largo sobre investigación de Alzheimer..."
    ... )
    """
    try:
        # Crear cliente de Azure OpenAI
        client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        
        # Prompt para generar el resumen
        prompt = f"""Genera un resumen conciso y completo del siguiente texto. 
El resumen debe capturar las ideas principales y ser informativo.

Texto:
{texto_entrada}

Resumen:"""
        
        # Llamar a la API
        print("Generando resumen...")
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "Eres un asistente experto en generar resúmenes concisos y precisos de textos científicos y médicos."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens,
            temperature=0.3
        )
        
        resumen = response.choices[0].message.content.strip()
        print("Resumen generado exitosamente.")
        return resumen
        
    except Exception as e:
        print(f"Error al generar resumen: {e}")
        raise


## Método 2: Extraer puntos más relevantes

Este método extrae los puntos más relevantes de un texto usando Azure OpenAI.


In [5]:
def extraer_puntos_relevantes(
    texto_entrada: str,
    endpoint: str = AZURE_OPENAI_ENDPOINT,
    api_key: str = AZURE_OPENAI_API_KEY,
    api_version: str = AZURE_OPENAI_API_VERSION,
    deployment_name: str = AZURE_OPENAI_DEPLOYMENT_NAME,
    max_tokens: int = 800,
    numero_puntos: int = 5
) -> List[str]:
    """
    Extrae los puntos más relevantes de un texto usando Azure OpenAI.
    
    Parámetros:
    -----------
    texto_entrada : str
        Texto del cual se desean extraer los puntos relevantes
    endpoint : str
        Endpoint de Azure OpenAI
    api_key : str
        Clave API de Azure OpenAI
    api_version : str
        Versión de la API de Azure OpenAI
    deployment_name : str
        Nombre del deployment en Azure OpenAI
    max_tokens : int
        Número máximo de tokens para la respuesta (por defecto 800)
    numero_puntos : int
        Número de puntos relevantes a extraer (por defecto 5)
    
    Retorna:
    --------
    List[str]
        Lista de puntos relevantes extraídos del texto
    
    Ejemplo:
    --------
    >>> puntos = extraer_puntos_relevantes(
    ...     texto_entrada="Texto sobre investigación de Alzheimer...",
    ...     numero_puntos=7
    ... )
    """
    try:
        # Crear cliente de Azure OpenAI
        client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        
        # Prompt para extraer puntos relevantes
        prompt = f"""Extrae los {numero_puntos} puntos más relevantes del siguiente texto.
Presenta cada punto en una línea separada, de forma clara y concisa.
Cada punto debe ser una idea principal o hallazgo importante.

Texto:
{texto_entrada}

Puntos relevantes:"""
        
        # Llamar a la API
        print(f"Extrayendo {numero_puntos} puntos relevantes...")
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "Eres un asistente experto en analizar textos científicos y médicos para extraer los puntos más relevantes e importantes."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens,
            temperature=0.3
        )
        
        contenido = response.choices[0].message.content.strip()
        
        # Procesar la respuesta para obtener una lista de puntos
        # Dividir por líneas y filtrar vacías
        puntos = [punto.strip() for punto in contenido.split('\n') if punto.strip()]
        
        # Limpiar numeración o viñetas si existen
        puntos_limpios = []
        for punto in puntos:
            # Remover numeración (1., 2., -., etc.)
            punto_limpio = re.sub(r'^[\d\-•\*]\s*', '', punto)
            puntos_limpios.append(punto_limpio)
        
        print(f"Se extrajeron {len(puntos_limpios)} puntos relevantes.")
        return puntos_limpios
        
    except Exception as e:
        print(f"Error al extraer puntos relevantes: {e}")
        raise


## Método 3: Extraer métricas (fecha y categorías)

Este método extrae métricas como fecha de publicación y categorías de Alzheimer de un texto usando Azure OpenAI.


In [6]:
def extraer_metricas(
    texto_entrada: str,
    tipos_alzheimer: List[str] = TIPOS_ALZHEIMER,
    endpoint: str = AZURE_OPENAI_ENDPOINT,
    api_key: str = AZURE_OPENAI_API_KEY,
    api_version: str = AZURE_OPENAI_API_VERSION,
    deployment_name: str = AZURE_OPENAI_DEPLOYMENT_NAME,
    max_tokens: int = 500
) -> Dict:
    """
    Extrae métricas como fecha de publicación y categorías de Alzheimer de un texto.
    
    Parámetros:
    -----------
    texto_entrada : str
        Texto del cual se desean extraer las métricas
    tipos_alzheimer : List[str]
        Lista de tipos de Alzheimer para identificar categorías
    endpoint : str
        Endpoint de Azure OpenAI
    api_key : str
        Clave API de Azure OpenAI
    api_version : str
        Versión de la API de Azure OpenAI
    deployment_name : str
        Nombre del deployment en Azure OpenAI
    max_tokens : int
        Número máximo de tokens para la respuesta (por defecto 500)
    
    Retorna:
    --------
    Dict
        Diccionario con las métricas extraídas:
        - fecha_publicacion: str (formato YYYY-MM-DD o None si no se encuentra)
        - categorias: List[str] (lista de tipos de Alzheimer encontrados)
        - fecha_original: str (fecha tal como aparece en el texto)
    
    Ejemplo:
    --------
    >>> metricas = extraer_metricas(
    ...     texto_entrada="Artículo publicado el 15 de marzo de 2024 sobre Alzheimer de inicio temprano..."
    ... )
    >>> print(metricas)
    {'fecha_publicacion': '2024-03-15', 'categorias': ['Alzheimer de inicio temprano'], 'fecha_original': '15 de marzo de 2024'}
    """
    try:
        # Crear cliente de Azure OpenAI
        client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        
        # Formatear lista de tipos de Alzheimer para el prompt
        lista_tipos = "\n".join([f"- {tipo}" for tipo in tipos_alzheimer])
        
        # Prompt para extraer métricas
        prompt = f"""Analiza el siguiente texto y extrae las siguientes métricas:

1. FECHA DE PUBLICACIÓN: Busca cualquier fecha de publicación, publicación, fecha del artículo, año de publicación, etc.
   Si encuentras una fecha, devuélvela en formato YYYY-MM-DD. Si no encuentras una fecha clara, devuelve null.

2. CATEGORÍAS DE ALZHEIMER: Identifica qué tipos de Alzheimer se mencionan o son relevantes en el texto.
   Solo selecciona de esta lista:
{lista_tipos}
   
   Si no se menciona ningún tipo específico, devuelve una lista vacía.

Devuelve la respuesta en formato JSON con esta estructura:
{{
    "fecha_publicacion": "YYYY-MM-DD o null",
    "fecha_original": "fecha tal como aparece en el texto o null",
    "categorias": ["tipo1", "tipo2", ...]
}}

Texto:
{texto_entrada}"""
        
        # Llamar a la API
        print("Extrayendo métricas...")
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "Eres un asistente experto en analizar textos científicos y médicos para extraer información estructurada como fechas y categorías. Siempre responde en formato JSON válido."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens,
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        
        contenido = response.choices[0].message.content.strip()
        
        # Parsear JSON
        try:
            metricas = json.loads(contenido)
            
            # Validar y limpiar fecha
            fecha_pub = metricas.get("fecha_publicacion")
            if fecha_pub and fecha_pub.lower() != "null":
                # Intentar validar formato de fecha
                try:
                    datetime.strptime(fecha_pub, "%Y-%m-%d")
                except ValueError:
                    # Si no es formato válido, intentar extraer año
                    año_match = re.search(r'\b(19|20)\d{2}\b', fecha_pub)
                    if año_match:
                        metricas["fecha_publicacion"] = f"{año_match.group()}-01-01"
                    else:
                        metricas["fecha_publicacion"] = None
            else:
                metricas["fecha_publicacion"] = None
            
            # Validar categorías
            categorias = metricas.get("categorias", [])
            if not isinstance(categorias, list):
                categorias = []
            
            # Filtrar categorías que estén en la lista permitida
            categorias_validas = [cat for cat in categorias if cat in tipos_alzheimer]
            metricas["categorias"] = categorias_validas
            
            print("Métricas extraídas exitosamente.")
            return metricas
            
        except json.JSONDecodeError as e:
            print(f"Error al parsear JSON: {e}")
            print(f"Contenido recibido: {contenido}")
            # Intentar extraer información manualmente
            return {
                "fecha_publicacion": None,
                "fecha_original": None,
                "categorias": []
            }
        
    except Exception as e:
        print(f"Error al extraer métricas: {e}")
        raise


## Ejemplos de uso

A continuación se muestran ejemplos de cómo usar los tres métodos.


### Ejemplo 1: Generar resumen


In [7]:
# Ejemplo: Generar resumen
texto_ejemplo = """
 La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo que afecta principalmente 
 a personas mayores. Investigaciones recientes publicadas en 2024 han demostrado que el Alzheimer 
 de inicio temprano puede manifestarse antes de los 65 años. Los estudios sugieren que factores 
 genéticos y ambientales juegan un papel importante en el desarrollo de la enfermedad.
 """
 
resumen = generar_resumen(
     texto_entrada=texto_ejemplo,
     max_tokens=300
)
print("Resumen generado:")
print(resumen)


Generando resumen...
Resumen generado exitosamente.
Resumen generado:
La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo que afecta principalmente a adultos mayores, aunque investigaciones recientes de 2024 indican que puede presentarse antes de los 65 años en casos de inicio temprano. Factores genéticos y ambientales influyen significativamente en su desarrollo.


### Ejemplo 2: Extraer puntos relevantes


In [9]:
# Ejemplo: Extraer puntos relevantes

puntos = extraer_puntos_relevantes(
     texto_entrada=texto_ejemplo,
     numero_puntos=5
)
print("Puntos relevantes:")
for i, punto in enumerate(puntos, 1):
    print(f"{i} {punto}")


Extrayendo 5 puntos relevantes...
Se extrajeron 5 puntos relevantes.
Puntos relevantes:
1 . La enfermedad de Alzheimer es un trastorno neurodegenerativo progresivo.
2 . Afecta principalmente a personas mayores.
3 . Investigaciones de 2024 muestran que el Alzheimer de inicio temprano puede aparecer antes de los 65 años.
4 . Factores genéticos influyen significativamente en el desarrollo del Alzheimer.
5 . Factores ambientales también contribuyen al riesgo de desarrollar la enfermedad.


### Ejemplo 3: Extraer métricas


In [10]:
# Ejemplo: Extraer métricas
 
metricas = extraer_metricas(
     texto_entrada=texto_ejemplo,
     tipos_alzheimer=TIPOS_ALZHEIMER
)
 
print("Métricas extraídas:")
print(f"Fecha de publicación: {metricas['fecha_publicacion']}")
print(f"Fecha original: {metricas.get('fecha_original', 'N/A')}")
print(f"Categorías: {', '.join(metricas['categorias']) if metricas['categorias'] else 'Ninguna'}")


Extrayendo métricas...
Métricas extraídas exitosamente.
Métricas extraídas:
Fecha de publicación: 2024-01-01
Fecha original: 2024
Categorías: Alzheimer de inicio temprano


### Ejemplo 4: Uso combinado de los tres métodos


In [13]:
#Procesar un artículo completo con los tres métodos

# Noticia de prueba:
# https://elpais.com/salud-y-bienestar/2025-12-03/silencios-que-alertan-por-que-la-perdida-auditiva-en-mayores-esta-relacionada-con-el-riesgo-de-alzheimer-y-como-prevenirlo.html

articulo_completo = """
Silencios que alertan: por qué la pérdida auditiva en mayores está relacionada con el riesgo de alzhéimer y cómo prevenirlo
Varios expertos aconsejan a los adultos hacerse revisiones periódicas y no esperar a tener problemas graves de comunicación para buscar solución


Un médico otorrinolaringólogo examina el oído de un paciente mayor con un otoscopio.
manassanant pamai (Getty Images)
Isabel Rubio
Isabel Rubio
03 DIC 2025 - 05:30 CET
Compartir en WhatsappCompartir en FacebookCompartir en TwitterCompartir en BlueskyCompartir en LinkedinCopiar enlace
27
Ir a los comentarios
Una de cada cuatro personas mayores de 60 años vive con una pérdida de audición discapacitante, según la Organización Mundial de la Salud (OMS). Las consecuencias pueden ir mucho más allá de las dificultades para seguir una conversación, hablar por teléfono o escuchar el timbre. La pérdida auditiva crónica y no tratada conlleva una amenaza que se gesta en el cerebro: el aumento de riesgo de deterioro cognitivo y demencia.

“La pérdida de audición adquirida en la adultez se asocia con un aumento del riesgo de deterioro cognitivo, demencia y, específicamente, enfermedad de Alzheimer”, explica Javier Camiña, neurólogo y vocal de la Sociedad Española de Neurología (SEN). La enfermedad de Alzheimer es la forma más común de demencia y puede representar entre un 60% y un 70% de los casos, como indica la OMS. Solo en España afecta a más de 800.000 personas. Camiña destaca que la pérdida auditiva puede ir asociada a un riesgo significativamente mayor de padecer esta enfermedad. En cualquier caso, por ahora se trata de una correlación, sin que se haya demostrado que la falta de audición cause la enfermedad neurológica.


Más información
Drogas Estados Unidos
“Muertes por desesperación”: la mortalidad global se desploma, pero repunta entre los jóvenes de Norteamérica por las drogas y los suicidios
Los investigadores intentan cuantificar la relación entre el deterioro cognitivo y la pérdida de audición. Un metanálisis publicado en la revista científica Ageing Research Reviews concluye que cada deterioro de 10 decibelios se asocia con un aumento del 16% en el riesgo de padecer demencia. Este hallazgo adquiere mayor relevancia en un contexto de envejecimiento poblacional y mayor longevidad: un estudio publicado en The Lancet Public Health proyecta que el número de personas con demencia aumentará de 57,4 millones en 2019 a 152,8 millones en 2050 en todo el mundo.


Gráfico elaborado con datos de la investigación publicada en ‘The Lancet Public Health’.
La Comisión Lancet reconoció la pérdida de audición como el principal factor de riesgo modificable para el deterioro cognitivo y la demencia. Hay varios mecanismos que podrían explicar cómo aumenta el deterioro cognitivo. Camiña menciona el aumento de la carga cognitiva: “La pérdida auditiva obliga a dedicar más recursos cognitivos para procesar el habla y los sonidos, lo que reduce la disponibilidad de estos recursos para otras funciones cognitivas como la memoria y la atención”. A ello se suma, según el experto, una atrofia acelerada en ciertas regiones del cerebro. Por ejemplo, aquellas implicadas en el procesamiento auditivo, el lenguaje y la memoria.

Además, la pérdida auditiva a menudo provoca aislamiento social. Algo que, según el neurólogo, disminuye la estimulación cognitiva y aumenta el riesgo de deterioro cognitivo, demencia y depresión. “En personas mayores, incrementa el efecto ‘soledad’ que puede ya padecer la persona y la inseguridad por no percibir los sonidos correctamente”, cuenta Francesc Carreño, responsable de Audiología en una conocida marca de audífonos.

Revisiones anuales a partir de los 60
Escuchar bien es muy importante. Tanto como cuidar la vista, la memoria o la movilidad, según Manuel Mozota Núñez, responsable del Grupo de Trabajo de Otorrinolaringología de la Sociedad Española de Médicos Generales y de Familia (SEMG). No solo ayuda a reducir el riesgo de deterioro cognitivo, sino que “es fundamental ante alarmas, timbres y bocinas”: “Una buena audición previene accidentes domésticos y accidentes en la calle”.

La pérdida auditiva en personas mayores aparece por una causa muy común que es la presbiacusia —la pérdida progresiva de la audición por desgaste y edad—. Así lo indica Carreño: “Esto hace que la mayoría de personas a partir de los 50 años empecemos a notar que hay ciertos sonidos que no oímos como antes o que empezamos a no entender con la misma fluidez en las conversaciones”.

“La degeneración del oído interno hace que las células ciliadas se vayan dañando y no se regeneren”, añade Mozota, que señala que también pueden producirse alteraciones en el nervio auditivo y en el cerebro. El experto destaca que la exposición prolongada al ruido, ciertos medicamentos ototóxicos y factores de salud como hipertensión o diabetes también pueden afectar a la audición.

“Si algo podemos hacer para ‘envejecer bien’, además de ejercicio físico, es cuidar y tratar nuestra salud auditiva”, explica Paula Sánchez, vicepresidenta de la Asociación Española de Audiología. Todos los expertos consultados aconsejan a los adultos someterse a revisiones auditivas periódicas cada uno o dos años —especialmente a partir de los 50 o 60 años—. “Sería conveniente después de los 60 años incluir la audición en los chequeos de salud igual que se revisan otras patologías”, opina Mozota. Corregir la pérdida auditiva a tiempo, por ejemplo con audífonos, evita que los problemas asociados empeoren.

Sánchez recomienda someterse a la primera revisión con un otorrinolaringólogo. Una vez hecha y siempre que no aparezcan nuevos síntomas ni problemas de salud relacionados con el oído, señala que los controles posteriores pueden realizarse también en centros o gabinetes especializados en audiología. Basta con hacer una búsqueda en Google para encontrar centros que ofrecen revisiones auditivas gratuitas.

Hay personas mayores que asumen como “normal” su pérdida de audición al haber alcanzado una determinada edad, según Sánchez. Pero ni es “normal” ni hay que esperar a tener problemas graves de comunicación para buscar solución. Los audioprotesistas suelen atender a personas de 60 años o mayores con pérdida auditiva avanzada. “Después de hacerles preguntas para conocer su caso, nos damos cuenta de que llevan tiempo con este problema y que han esperado a tener verdaderos problemas de comunicación para mirarlo”, explica Carreño.

A quienes usan audífonos o han tenido anteriormente alguna patología auditiva, Mozota les aconseja hacerse una revisión con más frecuencia —cada seis a 12 meses— para controlar la evolución. “Si aparecen acúfenos, dificultad para oír conversaciones, dolor, secreción por el conducto auditivo externo o mareos, debemos pedir cita con el médico de atención primaria o el otorrinolaringólogo”, añade.

Limitar la exposición al ruido
Para prevenir la pérdida auditiva, también es importante limitar la exposición al ruido. “Es poco probable que niveles de sonido inferiores a 80 dB causen daños auditivos. A medida que aumenta la intensidad del sonido, también aumenta la posibilidad de dañar tus oídos”, indica la OMS. Rubén Polo, presidente de la comisión de Otología de la Sociedad Española de Otorrinolaringología y Cirugía de Cabeza y Cuello (SEORL-CCC), señala que la exposición prolongada —ocho horas o más— a ruidos por encima de 85 decibelios puede producir daño auditivo.

Mozota da algunos datos para ponerlo en perspectiva: “Una conversación normal emite sonidos de unos 60 dB, mientras que un concierto en vivo o la música de una discoteca supera habitualmente los 100 decibelios”. La música alta con auriculares “suele rondar entre 95 y 105 decibelios, una sirena de ambulancia alcanza entre 110 y 120, el motor de un avión entre 130 y 140 y un disparo o petardo puede sobrepasar los 150 decibelios”. En entornos con ruido fuerte o prolongado, como eventos deportivos, conciertos o al utilizar herramientas, Sánchez recomienda usar tapones para los oídos. También aconseja no usar auriculares más de una hora al día y mantener el volumen por debajo del 60%.


Gráfico de niveles de decibelios de distintos sonidos realizado con datos de la OMS.
Tapones de cera y limpieza del oído
Los episodios transitorios de pérdida auditiva causados por tapones de cera no aumentan el riesgo de pérdida auditiva permanente ni de deterioro cognitivo, siempre que sean identificados y tratados a tiempo. Así lo indica Camiña, que reconoce que en casos excepcionales, si la obstrucción persiste sin tratamiento durante mucho tiempo, pueden surgir complicaciones.

Para cuidar la salud auditiva y evitar infecciones, es importante mantener una buena higiene. “No hay que lavar los oídos en casa”, explica Sánchez. Como explica la experta, el oído se “autogestiona”. O lo que es lo mismo, “nuestro conducto auditivo expulsa la suciedad hacia fuera”, explica Carreño. Es decir, la limpieza debe limitarse a la oreja: no hay que meter nada dentro del oído. Sánchez explica que basta con secar la oreja usando una toalla o incluso bastoncillos: “Están diseñados para limpiar los recovecos de nuestras orejas, no para meterlos en el conducto auditivo”.

Además de los bastoncillos, Mozota aconseja evitar introducir en el conducto auditivo horquillas, llaves o dedos: “Solo empujan el cerumen hacia dentro y pueden dañar el tímpano”. Tampoco recomienda aplicar aceite, alcohol ni ningún remedio casero. “Y no hagas nunca un lavado si tienes dolor”, añade. Si se acumula mucha cera, Mozota señala que se pueden usar gotas o sprays específicos para la higiene del oído, aplicándolos con la cabeza inclinada. Pero advierte que no conviene abusar de ellos. Si alguien necesita limpiar este conducto, debería acudir al centro de salud o a un otorrinolaringólogo.
"""

#1. Generar resumen
resumen = generar_resumen(articulo_completo)
print("=== RESUMEN ===")
print(resumen)
print("\n")

#2. Extraer puntos relevantes
puntos = extraer_puntos_relevantes(articulo_completo, numero_puntos=7)
print("=== PUNTOS RELEVANTES ===")
for i, punto in enumerate(puntos, 1):
    print(f"{i}. {punto}")
print("\n")

#3. Extraer métricas
metricas = extraer_metricas(articulo_completo, tipos_alzheimer=TIPOS_ALZHEIMER)
print("=== MÉTRICAS ===")
print(json.dumps(metricas, indent=2, ensure_ascii=False))


Generando resumen...
Resumen generado exitosamente.
=== RESUMEN ===
La pérdida auditiva afecta a uno de cada cuatro mayores de 60 años y está asociada con un mayor riesgo de deterioro cognitivo, demencia y especialmente Alzheimer, aunque la relación es de correlación y no causalidad. Estudios señalan que cada pérdida de 10 decibelios incrementa un 16% el riesgo de demencia, y la Comisión Lancet la identifica como el principal factor de riesgo modificable. Los mecanismos implicados incluyen sobrecarga cognitiva, atrofia cerebral y aislamiento social, que agravan el deterioro cognitivo y la depresión.

La causa más común es la presbiacusia, relacionada con el envejecimiento del oído interno, exposición al ruido, medicamentos y enfermedades como hipertensión o diabetes. Los expertos recomiendan revisiones auditivas periódicas a partir de los 50-60 años, y una mayor frecuencia en quienes usan audífonos o tienen antecedentes auditivos. Corregir la pérdida auditiva, por ejemplo con audífonos

## Notas importantes

- **Configuración de Azure OpenAI**: Asegúrate de tener configurado correctamente tu recurso de Azure OpenAI con un deployment activo.
- **API Key**: Nunca compartas tu API key en código público. Usa variables de entorno o Azure Key Vault.
- **Costos**: Cada llamada a la API consume tokens, lo que tiene un costo asociado.
- **Límites de tokens**: Ajusta `max_tokens` según tus necesidades y el modelo que uses.
- **Tipos de Alzheimer**: Puedes personalizar la lista `TIPOS_ALZHEIMER` según tus necesidades específicas.
- **Formato de fecha**: El método de métricas intenta normalizar fechas al formato YYYY-MM-DD, pero puede requerir ajustes según el formato de entrada.
